<a href="https://colab.research.google.com/github/mgoldenitz/student-substance-use-analysis/blob/main/sql/SQL_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

   # SQL Practice: Student Substance Use Data
   Three practice sessions using SQLite on the student survey dataset (12,744 rows): aggregation, filtering, joins, subqueries, CTEs and window functions. Companion to the Power BI report in this repository.

In [1]:
import pandas as pd, sqlite3
from google.colab import files
files.upload()   # choose student_addiction_dataset_test.csv
df = pd.read_csv('student_addiction_dataset_test.csv')
con = sqlite3.connect(':memory:')
df.to_sql('students', con, index=False)
q = lambda sql: pd.read_sql(sql, con)

q("SELECT * FROM students LIMIT 5")

Saving student_addiction_dataset_test.csv to student_addiction_dataset_test.csv


,Experimentation,Academic_Performance_Decline,Social_Isolation,Financial_Issues,Physical_Mental_Health_Problems,Legal_Consequences,Relationship_Strain,Risk_Taking_Behavior,Withdrawal_Symptoms,Denial_and_Resistance_to_Treatment,Addiction_Class
0,Yes,Yes,Yes,No,Yes,No,No,No,Yes,No,No
1,No,No,Yes,No,No,Yes,No,Yes,No,No,No
2,No,No,No,Yes,No,Yes,No,No,Yes,No,No
3,Yes,No,Yes,No,Yes,Yes,No,Yes,No,Yes,Yes
4,No,No,No,No,Yes,No,No,No,No,Yes,No


In [2]:
q("SELECT COUNT(*) AS total_students FROM students")

,total_students
0,12744


In [3]:
q("SELECT COUNT(*) AS addicted FROM students WHERE Addiction_Class = 'Yes'")

,addicted
0,2639


In [4]:
q("""
SELECT Risk_Taking_Behavior, COUNT(*) AS students
FROM students
GROUP BY Risk_Taking_Behavior
""")

,Risk_Taking_Behavior,students
0,None,613
1,No,8396
2,Yes,3735


In [5]:
q("SELECT COUNT(*) AS missing FROM students WHERE Experimentation IS NULL")

,missing
0,645


In [7]:
q("""
SELECT Withdrawal_Symptoms,
       COUNT(*) AS students,
       ROUND(AVG(CASE WHEN Addiction_Class = 'Yes' THEN 1.0 ELSE 0 END) * 100, 1) AS addiction_pct
FROM students
GROUP BY Withdrawal_Symptoms
""")

,Withdrawal_Symptoms,students,addiction_pct
0,None,653,20.8
1,No,8356,20.4
2,Yes,3735,21.4


In [8]:
con.executescript("""
DROP TABLE IF EXISTS scores;
DROP TABLE IF EXISTS risk_levels;

CREATE TABLE scores AS
SELECT rowid AS student_id,
       Addiction_Class,
       (CASE WHEN Experimentation='Yes' THEN 1 ELSE 0 END
      + CASE WHEN Academic_Performance_Decline='Yes' THEN 1 ELSE 0 END
      + CASE WHEN Social_Isolation='Yes' THEN 1 ELSE 0 END
      + CASE WHEN Financial_Issues='Yes' THEN 1 ELSE 0 END
      + CASE WHEN Physical_Mental_Health_Problems='Yes' THEN 1 ELSE 0 END
      + CASE WHEN Legal_Consequences='Yes' THEN 1 ELSE 0 END
      + CASE WHEN Relationship_Strain='Yes' THEN 1 ELSE 0 END
      + CASE WHEN Risk_Taking_Behavior='Yes' THEN 1 ELSE 0 END
      + CASE WHEN Withdrawal_Symptoms='Yes' THEN 1 ELSE 0 END
      + CASE WHEN Denial_and_Resistance_to_Treatment='Yes' THEN 1 ELSE 0 END) AS risk_score
FROM students;

CREATE TABLE risk_levels (min_score INT, max_score INT, risk_level TEXT);
INSERT INTO risk_levels VALUES (0,3,'Low'), (4,6,'Moderate'), (7,10,'High');
""")
q("SELECT * FROM scores LIMIT 5")

,student_id,Addiction_Class,risk_score
0,1,No,5
1,2,No,3
2,3,No,3
3,4,Yes,6
4,5,No,2


In [10]:
q("""
SELECT Social_Isolation, COUNT(*) AS students
FROM students
GROUP BY Social_Isolation
ORDER BY students DESC
""")

,Social_Isolation,students
0,No,8352
1,Yes,3715
2,None,677


In [11]:
q("""
SELECT Legal_Consequences, Financial_Issues, COUNT(*) AS students
FROM students
WHERE Legal_Consequences IS NOT NULL AND Financial_Issues IS NOT NULL
GROUP BY Legal_Consequences, Financial_Issues
HAVING COUNT(*) > 2000
ORDER BY students DESC
""")

,Legal_Consequences,Financial_Issues,students
0,No,No,5465
1,No,Yes,2468
2,Yes,No,2466


In [12]:
q("SELECT risk_score, COUNT(*) AS students FROM scores GROUP BY risk_score ORDER BY risk_score")

,risk_score,students
0,0,416
1,1,1666
2,2,3053
3,3,3347
4,4,2465
5,5,1256
6,6,434
7,7,94
8,8,13


In [13]:
q("SELECT ROUND(AVG(risk_score), 2) FROM scores")

,"ROUND(AVG(risk_score), 2)"
0,2.93


In [14]:
q("""
SELECT COUNT(*) AS above_average
FROM scores
WHERE risk_score > (SELECT AVG(risk_score) FROM scores)
""")

,above_average
0,7609


In [15]:
q("""
SELECT r.risk_level,
       COUNT(*) AS students,
       ROUND(AVG(CASE WHEN s.Addiction_Class = 'Yes' THEN 1.0 ELSE 0 END) * 100, 1) AS addiction_pct
FROM scores AS s
JOIN risk_levels AS r
  ON s.risk_score BETWEEN r.min_score AND r.max_score
GROUP BY r.risk_level
ORDER BY MIN(r.min_score)
""")

,risk_level,students,addiction_pct
0,Low,8482,20.6
1,Moderate,4155,21.1
2,High,107,17.8


In [16]:
q("""
WITH ranked AS (
  SELECT student_id, risk_score,
         RANK()       OVER (ORDER BY risk_score DESC) AS rnk,
         DENSE_RANK() OVER (ORDER BY risk_score DESC) AS dense_rnk,
         ROW_NUMBER() OVER (ORDER BY risk_score DESC, student_id) AS row_num
  FROM scores
)
SELECT * FROM ranked WHERE row_num BETWEEN 12 AND 16
""")

,student_id,risk_score,rnk,dense_rnk,row_num
0,9414,8,1,1,12
1,11581,8,1,1,13
2,19,7,14,2,14
3,36,7,14,2,15
4,309,7,14,2,16


In [17]:
q("""
SELECT risk_score,
       COUNT(*) AS students,
       SUM(COUNT(*)) OVER (ORDER BY risk_score) AS running_total,
       ROUND(100.0 * SUM(COUNT(*)) OVER (ORDER BY risk_score) / SUM(COUNT(*)) OVER (), 1) AS cumulative_pct
FROM scores
GROUP BY risk_score
ORDER BY risk_score
""")

,risk_score,students,running_total,cumulative_pct
0,0,416,416,3.3
1,1,1666,2082,16.3
2,2,3053,5135,40.3
3,3,3347,8482,66.6
4,4,2465,10947,85.9
5,5,1256,12203,95.8
6,6,434,12637,99.2
7,7,94,12731,99.9
8,8,13,12744,100.0


In [18]:
q("""
WITH ranked AS (
  SELECT student_id, Addiction_Class, risk_score,
         ROW_NUMBER() OVER (PARTITION BY Addiction_Class
                            ORDER BY risk_score DESC, student_id) AS rn
  FROM scores
)
SELECT Addiction_Class, student_id, risk_score
FROM ranked
WHERE rn <= 3
""")

,Addiction_Class,student_id,risk_score
0,No,280,8
1,No,1542,8
2,No,2174,8
3,Yes,1326,8
4,Yes,2247,8
5,Yes,3342,8


In [19]:
con.executescript("""
DROP TABLE IF EXISTS levels_incomplete;
CREATE TABLE levels_incomplete (min_score INT, max_score INT, risk_level TEXT);
INSERT INTO levels_incomplete VALUES (0,3,'Low'), (4,6,'Moderate');
""")

In [20]:
q("""
SELECT COUNT(*) AS students_kept
FROM scores s
JOIN levels_incomplete r ON s.risk_score BETWEEN r.min_score AND r.max_score
""")

,students_kept
0,12637


In [21]:
q("""
SELECT COALESCE(r.risk_level, 'Unmatched') AS risk_level, COUNT(*) AS students
FROM scores s
LEFT JOIN levels_incomplete r ON s.risk_score BETWEEN r.min_score AND r.max_score
GROUP BY COALESCE(r.risk_level, 'Unmatched')
""")

,risk_level,students
0,Low,8482
1,Moderate,4155
2,Unmatched,107


In [22]:
q("""
WITH rates AS (
  SELECT 'Experimentation' AS factor,
         AVG(CASE WHEN Experimentation='Yes' THEN 1.0 ELSE 0 END)*100 AS yes_pct
  FROM students WHERE Experimentation IS NOT NULL
  UNION ALL
  SELECT 'Relationship_Strain',
         AVG(CASE WHEN Relationship_Strain='Yes' THEN 1.0 ELSE 0 END)*100
  FROM students WHERE Relationship_Strain IS NOT NULL
  UNION ALL
  SELECT 'Withdrawal_Symptoms',
         AVG(CASE WHEN Withdrawal_Symptoms='Yes' THEN 1.0 ELSE 0 END)*100
  FROM students WHERE Withdrawal_Symptoms IS NOT NULL
)
SELECT factor, ROUND(yes_pct, 1) AS yes_pct,
       RANK() OVER (ORDER BY yes_pct DESC) AS rnk
FROM rates
""")

,factor,yes_pct,rnk
0,Relationship_Strain,31.2,1
1,Withdrawal_Symptoms,30.9,2
2,Experimentation,30.5,3
